# Лабораторная работа 4. SSTable
Заполняем кэш 10000 значениями, сбрасываем в SSTable, затем создаём новый ридер и проверяем GET.

In [1]:
import os
import random
import time
from sstable import Cache, SSTableReader

SST = 'data.sst'
if os.path.exists(SST):
    os.remove(SST)

## Заполняем кэш 10000 значениями и сбрасываем в SSTable

In [2]:
cache = Cache()
N = 10000
expected = {}
for i in range(N):
    k = f'key_{i:06d}'
    v = f'value_{i}_{random.randint(0, 1_000_000)}'
    cache.set(k, v)
    expected[k] = v
print('записей в кэше:', len(cache))

t0 = time.perf_counter()
cache.flush_to_sstable(SST)
print(f'flush_to_sstable: {time.perf_counter()-t0:.3f}s')
print('размер файла:', os.path.getsize(SST), 'байт')

записей в кэше: 10000
flush_to_sstable: 0.012s
размер файла: 351268 байт


## Структура файла: HEADER, DATA BLOCKS, SPARSE INDEX, FOOTER

In [3]:
import struct
from sstable import HEADER_FMT, HEADER_SIZE, FOOTER_FMT, FOOTER_SIZE, MAGIC, FOOTER_MAGIC

with open(SST, 'rb') as f:
    header = f.read(HEADER_SIZE)
    magic, entry_count, sparse_step, data_offset = struct.unpack(HEADER_FMT, header)
    print('HEADER:')
    print('  magic       =', magic)
    print('  entry_count =', entry_count)
    print('  sparse_step =', sparse_step)
    print('  data_offset =', data_offset)
    f.seek(-FOOTER_SIZE, os.SEEK_END)
    sparse_offset, sparse_count, fmagic = struct.unpack(FOOTER_FMT, f.read(FOOTER_SIZE))
    print('FOOTER:')
    print('  sparse_offset =', sparse_offset)
    print('  sparse_count  =', sparse_count)
    print('  magic         =', fmagic)

print('DATA BLOCKS:', data_offset, '..', sparse_offset, '(', sparse_offset-data_offset, 'байт )')
print('SPARSE INDEX:', sparse_offset, '..', os.path.getsize(SST)-FOOTER_SIZE)

HEADER:
  magic       = b'SSTB0001'
  entry_count = 10000
  sparse_step = 64
  data_offset = 24
FOOTER:
  sparse_offset = 347794
  sparse_count  = 157
  magic         = b'FTR00001'
DATA BLOCKS: 24 .. 347794 ( 347770 байт )
SPARSE INDEX: 347794 .. 351248


## Новый кэш читает SSTable и отвечает на GET

In [4]:
reader = SSTableReader(SST)
print('entry_count в файле:', reader.entry_count)

for k in ['key_000000', 'key_000001', 'key_004242', 'key_009999']:
    print(f'{k} -> {reader.get(k)}')

print('отсутствующий ключ ->', reader.get('key_999999'))
print('ключ вне диапазона ->', reader.get('zzz'))

entry_count в файле: 10000
key_000000 -> value_0_827538
key_000001 -> value_1_185895
key_004242 -> value_4242_332045
key_009999 -> value_9999_486998
отсутствующий ключ -> None
ключ вне диапазона -> None


## Полная проверка: каждый из 10000 ключей возвращает корректное значение

In [5]:
t0 = time.perf_counter()
mismatches = 0
for k, v in expected.items():
    if reader.get(k) != v:
        mismatches += 1
elapsed = time.perf_counter() - t0
print(f'{N} GET-ов за {elapsed:.2f}s -> {N/elapsed:.0f} ops/s')
print('несовпадений:', mismatches)
assert mismatches == 0
print('OK: все значения прочитаны корректно')

10000 GET-ов за 0.29s -> 34080 ops/s
несовпадений: 0
OK: все значения прочитаны корректно


In [6]:
if os.path.exists(SST):
    os.remove(SST)